<a href="https://colab.research.google.com/github/osmanle89739/SDC_Simtools_Project_Nadja_Saleh_Leila_Osman/blob/main/Simtools_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import simpy
import numpy as np

# -----------------------------
# Parameter
# -----------------------------
SIM_DAYS = 14
BEDS = 100
LAMBDA = 8                 # Ø Ankünfte pro Tag (Poisson)
LOS_SHAPE = 2.0            # Semi-Markov
LOS_SCALE = 3.0
MC_RUNS = 1000             # Monte-Carlo-Replikationen
SEED = 42

# -----------------------------
# Eine DES-Simulation (g(X))
# -----------------------------
def simulate_once(seed):
    rng = np.random.default_rng(seed)

    max_occupancy = 0
    overload = False

    def patient(env, hospital):
        nonlocal max_occupancy, overload

        with hospital.request() as req:
            yield req

            current = hospital.count
            max_occupancy = max(max_occupancy, current)

            if current > BEDS:
                overload = True

            # Semi-Markov: Aufenthaltsdauer
            los = rng.gamma(LOS_SHAPE, LOS_SCALE)
            yield env.timeout(los)

    def arrival_process(env, hospital):
        while env.now < SIM_DAYS:
            # Poisson-Prozess über exponentielle Zwischenankunftszeit
            interarrival = rng.exponential(1 / LAMBDA)
            yield env.timeout(interarrival)
            env.process(patient(env, hospital))

    # --- simpy Setup ---
    env = simpy.Environment()
    hospital = simpy.Resource(env, capacity=BEDS)

    env.process(arrival_process(env, hospital))
    env.run(until=SIM_DAYS)

    return overload, max_occupancy

# -----------------------------
# Monte-Carlo-Schleife
# -----------------------------
overloads = 0
max_loads = []

for i in range(MC_RUNS):
    overload, max_occ = simulate_once(SEED + i)
    max_loads.append(max_occ)
    if overload:
        overloads += 1

# -----------------------------
# Ergebnisse
# -----------------------------
probability_ok = 1 - overloads / MC_RUNS

print("Monte-Carlo-Ergebnisse")
print("----------------------")
print(f"Simulationsdauer: {SIM_DAYS} Tage")
print(f"Bettenkapazität: {BEDS}")
print(f"Monte-Carlo-Läufe: {MC_RUNS}")
print(f"Wahrscheinlichkeit, dass Betten ausreichen: {probability_ok:.2%}")
print(f"Ø maximale Belegung: {np.mean(max_loads):.1f} Betten")
print(f"95%-Quantil der Maximalbelegung: {np.quantile(max_loads, 0.95):.1f}")